# March Machine Learning Mania 2026 - Stage 2
## LightGBM + XGBoost + CatBoost Ensemble

**Features:** Elo (MOV + home advantage + regression), Four Factors, Massey Ordinals, GLM quality, SOS, momentum, conference strength, coach experience, seeds.  
**Metric:** Brier Score

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import linregress
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import os, gc, glob

KAGGLE = os.path.exists('/kaggle/input')
OUTPUT_DIR = '/kaggle/working/' if KAGGLE else ''

# Auto-detect data path on Kaggle (handles different folder structures)
if KAGGLE:
    candidates = glob.glob('/kaggle/input/**/MRegularSeasonCompactResults.csv', recursive=True)
    if candidates:
        DATA_DIR = os.path.dirname(candidates[0]) + '/'
    else:
        DATA_DIR = '/kaggle/input/march-machine-learning-mania-2026/'
else:
    DATA_DIR = 'march-machine-learning-mania-2026/'

print(f'Data dir: {DATA_DIR}')

MIN_SEASON = 2003
MIN_SEASON_W = 2010
CURRENT_SEASON = 2026
VAL_SEASONS = [2022, 2023, 2024, 2025]

ELO_INIT = 1500
ELO_K = 20
ELO_HOME = 100
ELO_WIDTH = 400
ELO_REVERT = 0.75

CLIP_LOW, CLIP_HIGH = 0.02, 0.98
SEED = 42
np.random.seed(SEED)

def load(name):
    return pd.read_csv(DATA_DIR + name)

print('Setup complete.')

## 1. Data Loading

In [ ]:
m_rs = load('MRegularSeasonCompactResults.csv')
m_detail = load('MRegularSeasonDetailedResults.csv')
m_tourney = load('MNCAATourneyCompactResults.csv')
m_seeds = load('MNCAATourneySeeds.csv')
m_conf = load('MTeamConferences.csv')
m_coaches = load('MTeamCoaches.csv')
m_secondary = load('MSecondaryTourneyCompactResults.csv')

w_rs = load('WRegularSeasonCompactResults.csv')
w_detail = load('WRegularSeasonDetailedResults.csv')
w_tourney = load('WNCAATourneyCompactResults.csv')
w_seeds = load('WNCAATourneySeeds.csv')
w_conf = load('WTeamConferences.csv')
w_secondary = load('WSecondaryTourneyCompactResults.csv')

sub = load('SampleSubmissionStage2.csv')
print(f'Submission rows: {len(sub)}')

## 2. Elo Ratings

In [ ]:
def compute_elo(rs, tourney, secondary):
    """Compute Elo ratings with MOV, home court, season regression."""
    games = pd.concat([
        rs[['Season','DayNum','WTeamID','WScore','LTeamID','LScore','WLoc']],
        tourney[['Season','DayNum','WTeamID','WScore','LTeamID','LScore','WLoc']],
        secondary[['Season','DayNum','WTeamID','WScore','LTeamID','LScore','WLoc']]
    ]).sort_values(['Season','DayNum']).reset_index(drop=True)

    elo = {}
    history = []  # (Season, DayNum, TeamID, Elo)
    prev_season = None

    for row in games.itertuples(index=False):
        s, day, w, ws, l, ls, wloc = row.Season, row.DayNum, row.WTeamID, row.WScore, row.LTeamID, row.LScore, row.WLoc
        if s != prev_season:
            for t in elo: elo[t] = ELO_REVERT * elo[t] + (1 - ELO_REVERT) * ELO_INIT
            prev_season = s
        elo.setdefault(w, ELO_INIT)
        elo.setdefault(l, ELO_INIT)

        we, le = elo[w], elo[l]
        if wloc == 'H': we += ELO_HOME
        elif wloc == 'A': le += ELO_HOME

        exp_w = 1.0 / (1.0 + 10.0 ** ((le - we) / ELO_WIDTH))
        k = ELO_K * np.log(abs(ws - ls) + 1)
        elo[w] += k * (1 - exp_w)
        elo[l] -= k * (1 - exp_w)

        if day <= 132:  # Regular season only for features
            history.append((s, day, w, elo[w]))
            history.append((s, day, l, elo[l]))

    hdf = pd.DataFrame(history, columns=['Season','DayNum','TeamID','Elo'])

    summary = hdf.groupby(['Season','TeamID'])['Elo'].agg(
        Elo_Last='last', Elo_Mean='mean', Elo_Max='max', Elo_Std='std'
    ).reset_index()
    summary['Elo_Std'] = summary['Elo_Std'].fillna(0)

    # Elo trend
    def trend(g):
        if len(g) < 3: return 0.0
        return linregress(range(len(g)), g['Elo'].values).slope
    tr = hdf.groupby(['Season','TeamID']).apply(trend).reset_index(name='Elo_Trend')
    summary = summary.merge(tr, on=['Season','TeamID'], how='left')
    summary['Elo_Trend'] = summary['Elo_Trend'].fillna(0)
    return summary

print('Computing Elo...')
m_elo = compute_elo(m_rs, m_tourney, m_secondary)
w_elo = compute_elo(w_rs, w_tourney, w_secondary)
elo_all = pd.concat([m_elo, w_elo], ignore_index=True)
print(f'Elo: {len(elo_all)} team-seasons')

## 3. Season Statistics (Vectorized)

In [ ]:
def compute_season_stats(detail, min_season):
    """Vectorized Four Factors computation from detailed box scores."""
    df = detail[detail['Season'] >= min_season].copy()
    otf = (40 + 5 * df['NumOT']) / 40

    # Build winner and loser records as two DataFrames, then concat
    # Winner perspective
    w = pd.DataFrame({
        'Season': df['Season'].values, 'TeamID': df['WTeamID'].values,
        'Score': df['WScore'].values / otf.values,
        'OppScore': df['LScore'].values / otf.values,
        'FGM': df['WFGM'].values / otf.values, 'FGA': df['WFGA'].values / otf.values,
        'FGM3': df['WFGM3'].values / otf.values, 'FGA3': df['WFGA3'].values / otf.values,
        'FTM': df['WFTM'].values / otf.values, 'FTA': df['WFTA'].values / otf.values,
        'OR': df['WOR'].values / otf.values, 'DR': df['WDR'].values / otf.values,
        'Ast': df['WAst'].values / otf.values, 'TO': df['WTO'].values / otf.values,
        'Stl': df['WStl'].values / otf.values, 'Blk': df['WBlk'].values / otf.values,
        'PF': df['WPF'].values / otf.values,
        'OppFGA': df['LFGA'].values / otf.values, 'OppFGM': df['LFGM'].values / otf.values,
        'OppFGM3': df['LFGM3'].values / otf.values, 'OppFGA3': df['LFGA3'].values / otf.values,
        'OppFTM': df['LFTM'].values / otf.values, 'OppFTA': df['LFTA'].values / otf.values,
        'OppOR': df['LOR'].values / otf.values, 'OppDR': df['LDR'].values / otf.values,
        'OppTO': df['LTO'].values / otf.values,
        'Win': 1
    })
    # Possessions
    w_poss = df['WFGA'].values - df['WOR'].values + df['WTO'].values + 0.475 * df['WFTA'].values
    l_poss = df['LFGA'].values - df['LOR'].values + df['LTO'].values + 0.475 * df['LFTA'].values
    w['Poss'] = (w_poss + l_poss) / 2 / otf.values

    # Loser perspective
    lo = pd.DataFrame({
        'Season': df['Season'].values, 'TeamID': df['LTeamID'].values,
        'Score': df['LScore'].values / otf.values,
        'OppScore': df['WScore'].values / otf.values,
        'FGM': df['LFGM'].values / otf.values, 'FGA': df['LFGA'].values / otf.values,
        'FGM3': df['LFGM3'].values / otf.values, 'FGA3': df['LFGA3'].values / otf.values,
        'FTM': df['LFTM'].values / otf.values, 'FTA': df['LFTA'].values / otf.values,
        'OR': df['LOR'].values / otf.values, 'DR': df['LDR'].values / otf.values,
        'Ast': df['LAst'].values / otf.values, 'TO': df['LTO'].values / otf.values,
        'Stl': df['LStl'].values / otf.values, 'Blk': df['LBlk'].values / otf.values,
        'PF': df['LPF'].values / otf.values,
        'OppFGA': df['WFGA'].values / otf.values, 'OppFGM': df['WFGM'].values / otf.values,
        'OppFGM3': df['WFGM3'].values / otf.values, 'OppFGA3': df['WFGA3'].values / otf.values,
        'OppFTM': df['WFTM'].values / otf.values, 'OppFTA': df['WFTA'].values / otf.values,
        'OppOR': df['WOR'].values / otf.values, 'OppDR': df['WDR'].values / otf.values,
        'OppTO': df['WTO'].values / otf.values,
        'Win': 0
    })
    lo['Poss'] = w['Poss'].values  # Same possessions

    both = pd.concat([w, lo], ignore_index=True)

    # Aggregate
    agg_cols = ['Score','OppScore','FGM','FGA','FGM3','FGA3','FTM','FTA',
                'OR','DR','Ast','TO','Stl','Blk','PF',
                'OppFGA','OppFGM','OppFGM3','OppFGA3','OppFTM','OppFTA',
                'OppOR','OppDR','OppTO','Poss','Win']
    agg = both.groupby(['Season','TeamID'])[agg_cols].mean().reset_index()
    agg.rename(columns={'Win': 'WinPct'}, inplace=True)

    # Derived stats
    agg['PointDiff'] = agg['Score'] - agg['OppScore']
    agg['OEff'] = agg['Score'] / agg['Poss'] * 100
    agg['DEff'] = agg['OppScore'] / agg['Poss'] * 100
    agg['NEff'] = agg['OEff'] - agg['DEff']
    agg['EFG'] = (agg['FGM'] + 0.5 * agg['FGM3']) / agg['FGA'].replace(0, 1)
    agg['TOR'] = agg['TO'] / agg['Poss'].replace(0, 1)
    agg['ORPCT'] = agg['OR'] / (agg['OR'] + agg['OppDR']).replace(0, 1)
    agg['FTR'] = agg['FTM'] / agg['FGA'].replace(0, 1)
    agg['OppEFG'] = (agg['OppFGM'] + 0.5 * agg['OppFGM3']) / agg['OppFGA'].replace(0, 1)
    agg['OppTOR'] = agg['OppTO'] / agg['Poss'].replace(0, 1)
    agg['OppORPCT'] = agg['OppOR'] / (agg['OppOR'] + agg['DR']).replace(0, 1)
    agg['OppFTR'] = agg['OppFTM'] / agg['OppFGA'].replace(0, 1)
    agg['FG3Pct'] = agg['FGM3'] / agg['FGA3'].replace(0, 1)
    agg['OppFG3Pct'] = agg['OppFGM3'] / agg['OppFGA3'].replace(0, 1)
    agg['AstRate'] = agg['Ast'] / agg['FGM'].replace(0, 1)

    keep = ['Season','TeamID','WinPct','PointDiff','Score','OppScore',
            'OEff','DEff','NEff','EFG','TOR','ORPCT','FTR',
            'OppEFG','OppTOR','OppORPCT','OppFTR',
            'FG3Pct','OppFG3Pct','AstRate','Stl','Blk','DR','Poss','FGA','OppFGA','PF']
    return agg[keep]

print('Computing season stats...')
stats_all = pd.concat([
    compute_season_stats(m_detail, MIN_SEASON),
    compute_season_stats(w_detail, MIN_SEASON_W)
], ignore_index=True)
print(f'Stats: {len(stats_all)} team-seasons, {len(stats_all.columns)-2} features')

## 4. Massey Ordinals

In [ ]:
massey_raw = pd.read_csv(DATA_DIR + 'MMasseyOrdinals.csv')
massey_raw = massey_raw[massey_raw['Season'] >= MIN_SEASON]

TOP_SYS = ['POM','SAG','MOR','DOK','KPK','BWE','STY','TRK','WIL','PGH','MAS','JNG','INC','HAS','EMK','DII']

# Keep only last ranking day per team-season-system
idx = massey_raw.groupby(['Season','TeamID','SystemName'])['RankingDayNum'].idxmax()
massey = massey_raw.loc[idx]

# Average across all systems
massey_avg = massey.groupby(['Season','TeamID'])['OrdinalRank'].agg(
    MasseyMean='mean', MasseyMedian='median', MasseyMin='min', MasseyStd='std'
).reset_index()
massey_avg['MasseyStd'] = massey_avg['MasseyStd'].fillna(0)

# Pivot top systems
top = massey[massey['SystemName'].isin(TOP_SYS)]
pivot = top.pivot_table(index=['Season','TeamID'], columns='SystemName', values='OrdinalRank')
pivot.columns = ['Massey_' + c for c in pivot.columns]
pivot = pivot.reset_index()

massey_feat = massey_avg.merge(pivot, on=['Season','TeamID'], how='left')
print(f'Massey: {len(massey_feat)} team-seasons, {len(massey_feat.columns)-2} features')

del massey_raw, massey
gc.collect()

## 5. SOS, Momentum, Conference Strength

In [ ]:
def compute_sos(rs, elo):
    w = rs[['Season','WTeamID','LTeamID']].rename(columns={'WTeamID':'TeamID','LTeamID':'OppID'})
    l = rs[['Season','LTeamID','WTeamID']].rename(columns={'LTeamID':'TeamID','WTeamID':'OppID'})
    opps = pd.concat([w, l])
    opps = opps.merge(elo[['Season','TeamID','Elo_Last']].rename(columns={'TeamID':'OppID','Elo_Last':'OppElo'}),
                      on=['Season','OppID'], how='left')
    opps['OppElo'] = opps['OppElo'].fillna(ELO_INIT)
    return opps.groupby(['Season','TeamID'])['OppElo'].mean().reset_index().rename(columns={'OppElo':'SOS'})

def compute_momentum(rs, n=10):
    rs_s = rs.sort_values(['Season','DayNum'])
    w = rs_s[['Season','DayNum','WTeamID']].assign(Win=1).rename(columns={'WTeamID':'TeamID'})
    l = rs_s[['Season','DayNum','LTeamID']].assign(Win=0).rename(columns={'LTeamID':'TeamID'})
    recs = pd.concat([w, l]).sort_values(['Season','TeamID','DayNum'])
    return recs.groupby(['Season','TeamID']).tail(n).groupby(['Season','TeamID'])['Win'].mean().reset_index().rename(columns={'Win':'Momentum'})

def compute_conf_strength(conf, elo):
    ce = conf.merge(elo[['Season','TeamID','Elo_Last']], on=['Season','TeamID'], how='left')
    ce['Elo_Last'] = ce['Elo_Last'].fillna(ELO_INIT)
    cs = ce.groupby(['Season','ConfAbbrev'])['Elo_Last'].mean().reset_index().rename(columns={'Elo_Last':'ConfStr'})
    return conf.merge(cs, on=['Season','ConfAbbrev'], how='left')[['Season','TeamID','ConfStr']]

sos_all = pd.concat([compute_sos(m_rs, m_elo), compute_sos(w_rs, w_elo)])
mom_all = pd.concat([compute_momentum(m_rs), compute_momentum(w_rs)])
conf_all = pd.concat([compute_conf_strength(m_conf, m_elo), compute_conf_strength(w_conf, w_elo)])
print('SOS, Momentum, Conference Strength computed.')

## 6. Coach Experience

In [ ]:
def compute_coach(coaches, tourney):
    cs = coaches.sort_values(['Season','TeamID','LastDayNum'])
    lc = cs.groupby(['Season','TeamID']).tail(1)[['Season','TeamID','CoachName']].copy()

    tt = set(zip(tourney['Season'], tourney['WTeamID'])) | set(zip(tourney['Season'], tourney['LTeamID']))
    lc['InTourney'] = lc.apply(lambda r: int((r['Season'], r['TeamID']) in tt), axis=1)

    lc = lc.sort_values('Season')
    lc['CoachExp'] = lc.groupby('CoachName')['InTourney'].cumsum()

    lc['PrevTeam'] = lc.groupby('CoachName')['TeamID'].shift(1)
    lc['PrevSeason'] = lc.groupby('CoachName')['Season'].shift(1)
    lc['Same'] = ((lc['TeamID'] == lc['PrevTeam']) & (lc['Season'] == lc['PrevSeason'] + 1)).astype(int)

    tenures = []
    t = 1
    for same in lc['Same']:
        t = t + 1 if same else 1
        tenures.append(t)
    lc['CoachTenure'] = tenures
    return lc[['Season','TeamID','CoachExp','CoachTenure']]

coach_feat = compute_coach(m_coaches, m_tourney)
print(f'Coach features: {len(coach_feat)} team-seasons')

## 7. Seeds & GLM Quality

In [ ]:
# Seeds
all_seeds = pd.concat([m_seeds, w_seeds])
all_seeds['SeedNum'] = all_seeds['Seed'].apply(lambda s: int(s[1:3]))
seeds_df = all_seeds[['Season','TeamID','SeedNum']]
print(f'Seeds: {len(seeds_df)}, 2026 seeds: {(seeds_df.Season==2026).sum()}')

In [ ]:
# GLM Quality
def compute_quality(rs, seeds, min_s):
    rs = rs[rs['Season'] >= min_s].copy()
    records = []
    for season in rs['Season'].unique():
        srs = rs[rs['Season'] == season]
        st = set(seeds[seeds['Season'] == season]['TeamID'])
        if not st:
            st = set(srs['WTeamID'].value_counts().head(68).index)
        rel = set(st)
        rel |= set(srs[srs['LTeamID'].isin(st)]['WTeamID'])

        teams = sorted(rel)
        t2i = {t: i for i, t in enumerate(teams)}
        n = len(teams)
        Xs, ys = [], []
        for r in srs.itertuples(index=False):
            wi, li = t2i.get(r.WTeamID, -1), t2i.get(r.LTeamID, -1)
            if wi == -1 and li == -1: continue
            x = np.zeros(n)
            if wi >= 0: x[wi] = 1
            if li >= 0: x[li] = -1
            Xs.append(x)
            ys.append(r.WScore - r.LScore)
        if len(Xs) < n: continue

        model = Ridge(alpha=1.0).fit(np.array(Xs), np.array(ys))
        for t, i in t2i.items():
            records.append({'Season': season, 'TeamID': t, 'Quality': model.coef_[i]})
    return pd.DataFrame(records)

print('Computing GLM quality...')
quality_all = pd.concat([
    compute_quality(m_rs, seeds_df, MIN_SEASON),
    compute_quality(w_rs, seeds_df, MIN_SEASON_W)
])
print(f'Quality: {len(quality_all)} team-seasons')

## 8. Build Unified Team Feature Table

In [ ]:
# Merge all features into a single team-season table
team_feat = elo_all.copy()

for df, name in [(stats_all, None), (massey_feat, None), (sos_all, None),
                  (mom_all, None), (conf_all, None), (quality_all, None),
                  (seeds_df, None), (coach_feat, None)]:
    team_feat = team_feat.merge(df, on=['Season','TeamID'], how='outer')

print(f'Unified feature table: {len(team_feat)} team-seasons, {len(team_feat.columns)-2} features')
print(f'2026 teams: {(team_feat.Season == 2026).sum()}')

## 9. Build Training Data (Vectorized Merge)

In [ ]:
def build_matchups(games_df, team_feat, is_train=True):
    """Build matchup features via vectorized merge."""
    if is_train:
        df = games_df.copy()
        df['T1'] = np.minimum(df['WTeamID'], df['LTeamID'])
        df['T2'] = np.maximum(df['WTeamID'], df['LTeamID'])
        df['Target'] = (df['WTeamID'] == df['T1']).astype(int)
        matchups = df[['Season','T1','T2','Target']].copy()
    else:
        matchups = games_df.copy()

    # Feature columns (exclude Season, TeamID)
    feat_cols = [c for c in team_feat.columns if c not in ['Season','TeamID']]

    # Merge T1 features
    t1_feats = team_feat.rename(columns={c: f'T1_{c}' for c in feat_cols})
    t1_feats = t1_feats.rename(columns={'TeamID': 'T1'})
    matchups = matchups.merge(t1_feats, on=['Season','T1'], how='left')

    # Merge T2 features
    t2_feats = team_feat.rename(columns={c: f'T2_{c}' for c in feat_cols})
    t2_feats = t2_feats.rename(columns={'TeamID': 'T2'})
    matchups = matchups.merge(t2_feats, on=['Season','T2'], how='left')

    # Difference features
    for c in feat_cols:
        t1c, t2c = f'T1_{c}', f'T2_{c}'
        if t1c in matchups.columns and t2c in matchups.columns:
            if matchups[t1c].dtype in ['float64','int64','float32','int32']:
                matchups[f'd_{c}'] = matchups[t1c] - matchups[t2c]

    # Gender indicator
    matchups['IsWomen'] = (matchups['T1'] >= 3000).astype(int)

    return matchups

# Training data from historical tournament games
print('Building training data...')
all_tourney = pd.concat([
    m_tourney[m_tourney['Season'] >= MIN_SEASON],
    w_tourney[w_tourney['Season'] >= MIN_SEASON_W]
])
train_df = build_matchups(all_tourney, team_feat, is_train=True)
print(f'Training: {len(train_df)} games, {len(train_df.columns)} columns')

In [ ]:
# Define feature columns
meta_cols = ['Season','T1','T2','Target']
feature_cols = [c for c in train_df.columns if c not in meta_cols]

# Drop features with >50% NaN
nan_pct = train_df[feature_cols].isna().mean()
feature_cols = nan_pct[nan_pct < 0.5].index.tolist()
print(f'Features: {len(feature_cols)}')

# Show feature groups
groups = {}
for f in feature_cols:
    p = f.split('_')[0]
    groups[p] = groups.get(p, 0) + 1
for g, c in sorted(groups.items(), key=lambda x: -x[1]):
    print(f'  {g}: {c}')

## 10. Model Training & Cross-Validation

In [ ]:
X_all = train_df[feature_cols].values.astype(np.float32)
y_all = train_df['Target'].values.astype(np.float32)
seasons = train_df['Season'].values

oof_preds = np.zeros(len(train_df))
best_iters = {'lgb': [], 'xgb': [], 'cat': []}

lgb_params = {
    'objective': 'binary', 'metric': 'mse', 'boosting_type': 'gbdt',
    'num_leaves': 31, 'learning_rate': 0.02, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_child_samples': 15,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'verbose': -1, 'seed': SEED,
}

xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'rmse',
    'max_depth': 5, 'learning_rate': 0.02, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'min_child_weight': 15,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'tree_method': 'hist', 'seed': SEED,
}

print('Time-based cross-validation...')
for vs in VAL_SEASONS:
    tr_mask = seasons < vs
    va_mask = seasons == vs
    if va_mask.sum() == 0: continue

    Xtr, ytr = X_all[tr_mask].copy(), y_all[tr_mask]
    Xva, yva = X_all[va_mask].copy(), y_all[va_mask]

    # Impute NaN with training median
    medians = np.nanmedian(Xtr, axis=0)
    medians = np.where(np.isnan(medians), 0, medians)
    for j in range(Xtr.shape[1]):
        Xtr[np.isnan(Xtr[:, j]), j] = medians[j]
        Xva[np.isnan(Xva[:, j]), j] = medians[j]

    preds = np.zeros(va_mask.sum())

    # LightGBM
    m = lgb.train(lgb_params, lgb.Dataset(Xtr, ytr), num_boost_round=2000,
                  valid_sets=[lgb.Dataset(Xva, yva)],
                  callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
    best_iters['lgb'].append(m.best_iteration)
    preds += m.predict(Xva) / 3

    # XGBoost
    m2 = xgb.train(xgb_params, xgb.DMatrix(Xtr, ytr, feature_names=feature_cols),
                   num_boost_round=2000, evals=[(xgb.DMatrix(Xva, yva, feature_names=feature_cols), 'v')],
                   early_stopping_rounds=200, verbose_eval=0)
    best_iters['xgb'].append(m2.best_iteration)
    preds += m2.predict(xgb.DMatrix(Xva, feature_names=feature_cols)) / 3

    # CatBoost
    m3 = CatBoostClassifier(iterations=2000, learning_rate=0.02, depth=5,
                            min_data_in_leaf=15, l2_leaf_reg=3.0, subsample=0.8,
                            random_seed=SEED, verbose=0, eval_metric='BrierScore',
                            use_best_model=True)
    m3.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=200)
    best_iters['cat'].append(m3.best_iteration_)
    preds += m3.predict_proba(Xva)[:, 1] / 3

    preds = np.clip(preds, CLIP_LOW, CLIP_HIGH)
    oof_preds[va_mask] = preds
    print(f'  {vs}: Brier={brier_score_loss(yva, preds):.6f} (n={va_mask.sum()})')

val_mask = np.isin(seasons, VAL_SEASONS)
print(f'\nOverall OOF Brier: {brier_score_loss(y_all[val_mask], oof_preds[val_mask]):.6f}')
print(f'Best iters - LGB: {best_iters["lgb"]}, XGB: {best_iters["xgb"]}, CAT: {best_iters["cat"]}')

In [ ]:
# Retrain on full data
print('Retraining on full data...')
Xf = X_all.copy()
yf = y_all.copy()
global_medians = np.nanmedian(Xf, axis=0)
global_medians = np.where(np.isnan(global_medians), 0, global_medians)
for j in range(Xf.shape[1]):
    Xf[np.isnan(Xf[:, j]), j] = global_medians[j]

lgb_n = int(np.mean(best_iters['lgb']) * 1.1)
lgb_final = lgb.train(lgb_params, lgb.Dataset(Xf, yf), num_boost_round=lgb_n)

xgb_n = int(np.mean(best_iters['xgb']) * 1.1)
xgb_final = xgb.train(xgb_params, xgb.DMatrix(Xf, yf, feature_names=feature_cols), num_boost_round=xgb_n)

cat_n = int(np.mean(best_iters['cat']) * 1.1)
cat_final = CatBoostClassifier(iterations=cat_n, learning_rate=0.02, depth=5,
                                min_data_in_leaf=15, l2_leaf_reg=3.0, subsample=0.8,
                                random_seed=SEED, verbose=0)
cat_final.fit(Xf, yf)

print(f'Trained - LGB: {lgb_n}, XGB: {xgb_n}, CAT: {cat_n} rounds')

In [ ]:
# Feature importance
imp = pd.DataFrame({'feature': feature_cols,
                     'importance': lgb_final.feature_importance(importance_type='gain')})
print('Top 25 features (LGB gain):')
print(imp.sort_values('importance', ascending=False).head(25).to_string(index=False))

## 11. Stage 2 Predictions

In [ ]:
# Parse submission
test = sub.copy()
parts = test['ID'].str.split('_', expand=True).astype(int)
test['Season'] = parts[0]
test['T1'] = parts[1]
test['T2'] = parts[2]

# Build features via merge
test_feat = build_matchups(test[['Season','T1','T2']], team_feat, is_train=False)
test_feat['ID'] = test['ID'].values

# Ensure all feature columns exist
for c in feature_cols:
    if c not in test_feat.columns:
        test_feat[c] = np.nan

X_test = test_feat[feature_cols].values.astype(np.float32)
for j in range(X_test.shape[1]):
    X_test[np.isnan(X_test[:, j]), j] = global_medians[j]

print(f'Test shape: {X_test.shape}')

In [ ]:
# Ensemble predictions
p1 = lgb_final.predict(X_test)
p2 = xgb_final.predict(xgb.DMatrix(X_test, feature_names=feature_cols))
p3 = cat_final.predict_proba(X_test)[:, 1]

pred = np.clip((p1 + p2 + p3) / 3, CLIP_LOW, CLIP_HIGH)

print(f'Pred stats: mean={pred.mean():.4f}, std={pred.std():.4f}, '
      f'min={pred.min():.4f}, max={pred.max():.4f}')

In [ ]:
# Create submission
submission = pd.DataFrame({'ID': test_feat['ID'], 'Pred': pred})
submission.to_csv(OUTPUT_DIR + 'submission.csv', index=False)

# Sanity checks
expected = load('SampleSubmissionStage2.csv')
assert len(submission) == len(expected), f'Row count: {len(submission)} vs {len(expected)}'
assert list(submission.columns) == ['ID', 'Pred']
assert (submission['Pred'] >= 0).all() and (submission['Pred'] <= 1).all()
assert submission['ID'].tolist() == expected['ID'].tolist()

print(f'Submission saved: {len(submission)} rows')
print(submission.head())
print('\nAll checks passed!')

## 12. Prediction EDA - Who Wins What?
Let's make sense of our predictions by mapping team IDs to names and showing power rankings, key matchups, and championship odds.

In [ ]:
# Build team name lookup and fast probability dictionary
teams_m = load('MTeams.csv')
teams_w = load('WTeams.csv')
TEAM_NAMES = dict(zip(teams_m['TeamID'], teams_m['TeamName']))
TEAM_NAMES.update(dict(zip(teams_w['TeamID'], teams_w['TeamName'])))

# Fast probability lookup: (lower_id, higher_id) -> P(lower wins)
prob_lookup = {}
for _, r in submission.iterrows():
    parts = r['ID'].split('_')
    prob_lookup[(int(parts[1]), int(parts[2]))] = r['Pred']

def win_prob(t1, t2):
    """Get P(t1 beats t2)."""
    lo, hi = min(t1, t2), max(t1, t2)
    p = prob_lookup.get((lo, hi), 0.5)
    return p if t1 == lo else 1 - p

def name(tid):
    return TEAM_NAMES.get(tid, f'Team {tid}')

print(f'Loaded {len(TEAM_NAMES)} team names, {len(prob_lookup)} matchup probabilities')

In [ ]:
# --- MEN'S POWER RANKINGS ---
# Average predicted win probability vs ALL other men's opponents
men_ids = sorted(set(t for k in prob_lookup for t in k if 1000 <= t < 2000))
men_strength = {}
for tid in men_ids:
    probs = [win_prob(tid, opp) for opp in men_ids if opp != tid]
    men_strength[tid] = np.mean(probs)

men_ranked = sorted(men_strength.items(), key=lambda x: -x[1])

print("=" * 60)
print(f"{'#':>3}  {'TEAM':<24} {'AVG WIN PROB':>12}  {'ELO':>7}")
print("=" * 60)
print("  MEN'S TOP 25")
print("-" * 60)
elo_lookup = dict(zip(zip(elo_all['Season'], elo_all['TeamID']), elo_all['Elo_Last']))
for i, (tid, wp) in enumerate(men_ranked[:25]):
    elo_val = elo_lookup.get((2026, tid), 0)
    print(f"  {i+1:2d}. {name(tid):<24s} {wp:>10.1%}    {elo_val:7.1f}")

In [ ]:
# --- WOMEN'S POWER RANKINGS ---
women_ids = sorted(set(t for k in prob_lookup for t in k if t >= 3000))
women_strength = {}
for tid in women_ids:
    probs = [win_prob(tid, opp) for opp in women_ids if opp != tid]
    women_strength[tid] = np.mean(probs)

women_ranked = sorted(women_strength.items(), key=lambda x: -x[1])

print("=" * 60)
print(f"{'#':>3}  {'TEAM':<24} {'AVG WIN PROB':>12}  {'ELO':>7}")
print("=" * 60)
print("  WOMEN'S TOP 25")
print("-" * 60)
for i, (tid, wp) in enumerate(women_ranked[:25]):
    elo_val = elo_lookup.get((2026, tid), 0)
    print(f"  {i+1:2d}. {name(tid):<24s} {wp:>10.1%}    {elo_val:7.1f}")

In [ ]:
# --- KEY HEAD-TO-HEAD MATCHUPS (Men's Top 8) ---
top8_m = [r[0] for r in men_ranked[:8]]

print("=" * 65)
print("MEN'S HEAD-TO-HEAD: Top 8 Teams vs Each Other")
print("=" * 65)
for i in range(len(top8_m)):
    for j in range(i+1, len(top8_m)):
        t1, t2 = top8_m[i], top8_m[j]
        p = win_prob(t1, t2)
        winner = name(t1) if p > 0.5 else name(t2)
        print(f"  {name(t1):<16s} vs {name(t2):<16s}  -->  {winner} wins ({max(p,1-p):.0%})")

print()
print("=" * 65)
print("WOMEN'S HEAD-TO-HEAD: Top 8 Teams vs Each Other")
print("=" * 65)
top8_w = [r[0] for r in women_ranked[:8]]
for i in range(len(top8_w)):
    for j in range(i+1, len(top8_w)):
        t1, t2 = top8_w[i], top8_w[j]
        p = win_prob(t1, t2)
        winner = name(t1) if p > 0.5 else name(t2)
        print(f"  {name(t1):<16s} vs {name(t2):<16s}  -->  {winner} wins ({max(p,1-p):.0%})")

In [ ]:
# --- CHAMPIONSHIP ODDS (Monte Carlo Simulation) ---
# Simulate a 16-team single-elimination bracket 50K times

def simulate_championship(top16_ids, n_sims=50000):
    np.random.seed(42)
    champ = {t: 0 for t in top16_ids}
    final = {t: 0 for t in top16_ids}
    rands = np.random.random((n_sims, 15))
    for sim in range(n_sims):
        bracket = list(top16_ids)
        ri = 0
        while len(bracket) > 1:
            nxt = []
            for k in range(0, len(bracket), 2):
                if k+1 >= len(bracket):
                    nxt.append(bracket[k]); continue
                p = win_prob(bracket[k], bracket[k+1])
                nxt.append(bracket[k] if rands[sim, ri] < p else bracket[k+1])
                if len(bracket) == 2:
                    final[bracket[k]] += 1
                    final[bracket[k+1]] += 1
                ri += 1
            bracket = nxt
        champ[bracket[0]] += 1
    return champ, final

print("=" * 55)
print("MEN'S CHAMPIONSHIP ODDS (50K simulations)")
print("=" * 55)
print(f"  {'TEAM':<24s} {'CHAMPION':>10}  {'FINALS':>8}")
print("-" * 55)
top16_m = [r[0] for r in men_ranked[:16]]
m_champ, m_final = simulate_championship(top16_m)
for tid, c in sorted(m_champ.items(), key=lambda x: -x[1]):
    cp = c / 50000 * 100
    fp = m_final[tid] / 50000 * 100
    if cp >= 0.5:
        print(f"  {name(tid):<24s} {cp:>8.1f}%  {fp:>7.1f}%")

print()
print("=" * 55)
print("WOMEN'S CHAMPIONSHIP ODDS (50K simulations)")
print("=" * 55)
print(f"  {'TEAM':<24s} {'CHAMPION':>10}  {'FINALS':>8}")
print("-" * 55)
top16_w = [r[0] for r in women_ranked[:16]]
w_champ, w_final = simulate_championship(top16_w)
for tid, c in sorted(w_champ.items(), key=lambda x: -x[1]):
    cp = c / 50000 * 100
    fp = w_final[tid] / 50000 * 100
    if cp >= 0.5:
        print(f"  {name(tid):<24s} {cp:>8.1f}%  {fp:>7.1f}%")

In [ ]:
# --- BIGGEST UPSETS: Games closest to 50/50 among top teams ---
print("=" * 65)
print("CLOSEST MATCHUPS (Coin-flip games among Top 20 men's teams)")
print("=" * 65)
top20_m = [r[0] for r in men_ranked[:20]]
close_games = []
for i in range(len(top20_m)):
    for j in range(i+1, len(top20_m)):
        t1, t2 = top20_m[i], top20_m[j]
        p = win_prob(t1, t2)
        close_games.append((t1, t2, p, abs(p - 0.5)))

close_games.sort(key=lambda x: x[3])
print(f"  {'TEAM 1':<20s} {'TEAM 2':<20s} {'WIN PROB':>10}")
print("-" * 65)
for t1, t2, p, _ in close_games[:15]:
    fav = name(t1) if p > 0.5 else name(t2)
    dog = name(t2) if p > 0.5 else name(t1)
    fp = max(p, 1-p)
    print(f"  {fav:<20s} {dog:<20s} {fp:>8.1%}")

print()
print("=" * 65)
print("BIGGEST MISMATCHES (Most lopsided among Top 20 men's)")  
print("=" * 65)
close_games.sort(key=lambda x: -x[3])
print(f"  {'FAVORITE':<20s} {'UNDERDOG':<20s} {'WIN PROB':>10}")
print("-" * 65)
for t1, t2, p, _ in close_games[:10]:
    fav = name(t1) if p > 0.5 else name(t2)
    dog = name(t2) if p > 0.5 else name(t1)
    fp = max(p, 1-p)
    print(f"  {fav:<20s} {dog:<20s} {fp:>8.1%}")

In [ ]:
# --- PREDICTION DISTRIBUTION ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Men's predictions
men_preds = submission[submission['ID'].str.startswith('2026_1')]['Pred']
axes[0].hist(men_preds, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='50/50')
axes[0].set_title("Men's Prediction Distribution", fontsize=14)
axes[0].set_xlabel('P(Team 1 wins)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Women's predictions
women_preds = submission[submission['ID'].str.startswith('2026_3')]['Pred']
axes[1].hist(women_preds, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='50/50')
axes[1].set_title("Women's Prediction Distribution", fontsize=14)
axes[1].set_xlabel('P(Team 1 wins)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Men's: mean={men_preds.mean():.3f}, std={men_preds.std():.3f}")
print(f"Women's: mean={women_preds.mean():.3f}, std={women_preds.std():.3f}")

In [ ]:
# --- READABLE SUBMISSION SAMPLE ---
# Show what the submission actually means in plain English
print("=" * 75)
print("SAMPLE PREDICTIONS (translated to team names)")
print("=" * 75)
print(f"  {'ID':<20s} {'MATCHUP':<40s} {'PRED':>6}")
print("-" * 75)

sample_ids = submission.sample(20, random_state=42).sort_values('Pred', ascending=False)
for _, row in sample_ids.iterrows():
    parts = row['ID'].split('_')
    t1, t2 = int(parts[1]), int(parts[2])
    n1, n2 = name(t1), name(t2)
    p = row['Pred']
    if p > 0.5:
        matchup = f"{n1} beats {n2}"
    else:
        matchup = f"{n2} beats {n1}"
    print(f"  {row['ID']:<20s} {matchup:<40s} {max(p,1-p):>5.1%}")

print()
print("TRANSLATION: Each row predicts P(lower TeamID wins).")
print("Example: '2026_1101_1102, 0.63' means Team 1101 has a 63% chance of beating Team 1102.")